## 1장 1강 : LLM 애플리케이션의 입력과 출력 구조

### 의존 패키지 설치
```
uv add ipykernel langchain langchain-core langchain-ollama python-dotenv
```

### 3. LCEL 파이프라인 실습

#### 3.1 환경 변수 로드 및 패키지 불러오기

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

#### 3.2 ChatPromptTemplate으로 입력 템플릿 조립하기

시스템 역할과 사용자 질문 변수({user_question})를 담은 템플릿 생성<br>
system: "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."<br>
user: "{user_question}"

In [5]:
# system : 역할, 상황, 제한 조건, 예시 -> 시스템 메세지, SystemMessage(...)
# user : 사용자의 질의 - HumanMessage(...)
# assistant : AI의 답변 - AIMessage(...)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."),
    ("user", "{user_question}")
])



템플릿에 텍스트용 질문 데이터를 주입하여 결과를 확인

user_question: "프로그래밍에서 '변수'가 무엇인가요?"

In [9]:
sample_prompt = prompt_template.invoke({
    "user_question": "프로그래밍에서 '변수'가 무엇인가요?"
})

sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content="프로그래밍에서 '변수'가 무엇인가요?", additional_kwargs={}, response_metadata={})])

#### 3.3 ChatOllama로 mistral 모델 호출하기

ChatOllama 모델 인스턴스 생성

In [10]:
model = ChatOllama(
    model = "mistral",
    #base_url="http://localhost:11434" # Ollama 서버 주소
)

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral')

In [11]:
response = model.invoke(sample_prompt)

response

AIMessage(content=' 변수는 프로그래밍 中의 상자와 같습니다. 이름 붙인 상자에 데이터를 저장하고, 그 데이터를 프로그램 내에서 필요할 때 언제든지 꺼내올 수 있습니다. 예를 들어, "친구의 이름"은 변수일 수 있습니다. 프로그램이 "친구의 이름"을 필요로 할 때마다, 이 변수에서 친구의 이름을 얻을 수 있습니다.', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T05:46:37.5978058Z', 'done': True, 'done_reason': 'stop', 'total_duration': 45789030200, 'load_duration': 23596639400, 'prompt_eval_count': 106, 'prompt_eval_duration': 18186037000, 'eval_count': 178, 'eval_duration': 3998212000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084b3-81bc-7610-8c35-0d098f35c3cc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 178, 'total_tokens': 284})

앞서 만든 sample_prompt를 모델에 직접 전달하여 실행

#### 3.4 StrOutputParser로 순수 텍스트만 추출하기

문자열 출력 파서 생성

In [12]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response 객체에서 순수 텍스트만 추출

In [13]:
text = parser.invoke(response)

text

' 변수는 프로그래밍 中의 상자와 같습니다. 이름 붙인 상자에 데이터를 저장하고, 그 데이터를 프로그램 내에서 필요할 때 언제든지 꺼내올 수 있습니다. 예를 들어, "친구의 이름"은 변수일 수 있습니다. 프로그램이 "친구의 이름"을 필요로 할 때마다, 이 변수에서 친구의 이름을 얻을 수 있습니다.'

#### 3.5 LCEL 파이프(|) 연산자로 완전한 체인 결합 및 실행하기

파이프(|) 연산자를 사용해 3개 컴포넌트를 하나의 파이프라인으로 연결

In [14]:
chain = prompt_template | model | parser

새로운 질문으로 파이프라인 전체 실행

In [17]:
res = chain.invoke({
        "user_question":"클래스(class)에 대해 알기 쉬운 비유를 통해 설명하시오."
})

res

' 클래스는 사물의 蓝핵을 생각하면 편하습니다. 그림 속 고양이와 개 두 마리가 있다고 생각해 보세요. 이 두 가지는 모두 동물이지만 특성이나 행동이 서로 다릅니다. 그런 다음, 고양이와 개를 기반으로 새로운 동물 클래스를 만듭니다. 이 새로운 클래스는 공통 특성과 행동(예: 걸을 수 있음, 짖음)을 정의합니다. 그런 다음, 이 클래스를 사용하여 고양이와 개와 같은 새로운 객체(사물)를 만들 수 있습니다. 이를 통해, 코드에서 공통 특성과 행동을 한 번만 정의하고, 이를 사용하여 여러 개체를 만들 수 있습니다.'

### 4. 프롬프트 엔지니어링
#### 4.1 프롬프트 엔지니어링의 3대 핵심 역할
- 목표 명확화
- 제약 조건 부여
- 출력 규격 표준화

#### 4.2 AI 성능 확장 4단계 비교
- Prompt Engineering
- RAG (검색 증강 생성)
- Fine-tuning (미세 조정)
- AI Agent (에이전트)